In [ ]:
from pathlib import Path
import mne
import numpy as np
import json
import csv
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()

# Nach oben gehen, bis das Datenverzeichnis gefunden wird
while not (PROJECT_ROOT / "data" / "raw" / "ds005363-download").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Projektverzeichnis nicht gefunden.")

    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "ds005363-download"

print("Arbeitsverzeichnis:", Path.cwd())
print("Datenverzeichnis:", DATA_ROOT)

def read_tsv(path: Path) -> list[dict[str, str]]:
    """Liest eine TSV-Datei robust gegen ein UTF-8-BOM ein."""

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file, delimiter="\t"))

def first_matching(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Keine Datei mit Muster {pattern!r} in {directory}")
    return matches[0]

def subject_directory(data_root: Path, subject: str) -> Path:
    subject_label = subject if subject.startswith("sub-") else f"sub-{subject}"
    directory = data_root / subject_label / "ses-01" / "eeg"
    if not directory.is_dir():
        raise FileNotFoundError(f"EEG-Verzeichnis nicht gefunden: {directory}")
    return directory

def read_data(subject: str, duration: float, data_root: Path, output_root: Path): 
    eeg_dir = subject_directory(data_root, subject)

    vhdr_path = first_matching(eeg_dir, "*_eeg.vhdr")
    json_path = first_matching(eeg_dir, "*_eeg.json")
    channels_path = first_matching(eeg_dir, "*_channels.tsv")
    events_path = first_matching(eeg_dir, "*_events.tsv")

    with json_path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)
    channels = read_tsv(channels_path)
    events = read_tsv(events_path)

    print(f"Lese BrainVision-Aufnahme ein: {vhdr_path}")
    raw = mne.io.read_raw_brainvision(vhdr_path, preload=False, verbose="ERROR")
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    if len(eeg_picks) == 0:
        raise RuntimeError("Die Aufnahme enthält keine als EEG markierten Kanäle.")

    return raw, channels, events



Arbeitsverzeichnis: /home/bened/bachelor-thesis/notebooks/BunDLe-Net/notebooks
Datenverzeichnis: /home/bened/bachelor-thesis/data/raw/ds005363-download


In [ ]:
try:
    raw, channels, events = read_data(
        subject="101",
        duration=60.0,
        data_root=DATA_ROOT,
        output_root=PROJECT_ROOT / "results",
    )

except FileNotFoundError as exc:
    raise SystemExit(f"Datei nicht verfügbar: {exc}") from exc
except RuntimeError as exc:
    raise SystemExit(f"Analyse konnte nicht durchgeführt werden: {exc}") from exc

batch_size = 1024
win = 15
block_size = 16*batch_size-win

stimulus_to_state = {}

for code in range(1, 17):
    stimulus_to_state[code] = 1  # Gesicht

for code in range(17, 33):
    stimulus_to_state[code] = 2  # Tier

for code in range(33, 49):
    stimulus_to_state[code] = 3  # Ort

for code in range(49, 65):
    stimulus_to_state[code] = 4  # Objekt

for code in range(65, 70):
    stimulus_to_state[code] = 5

def extract_s(trial_type):
    if not trial_type.startswith("Stimulus/S"):
        return None

    number = trial_type.removeprefix("Stimulus/S").strip()
    return int(number)

labels = []

for event in events:
    code = extract_s(event["trial_type"])

    if code in stimulus_to_state:
        label = stimulus_to_state[code]
    elif code == 199 or code in range(101, 109):
        label = 0  
    else:
        continue 

    sample = int(event["sample"])
        
    labels.append((sample, label))

import queue
import threading

def prep_data(X, B, win=15):
    if win < 1 or len(X) <= win:
        raise ValueError("X must contain more samples than the positive window length.")
    if len(X) != len(B):
        raise ValueError("X and B must have the same number of samples.")
    # Read-only views share the original recording, including overlapping pairs.
    windows = np.lib.stride_tricks.sliding_window_view(
        X, window_shape=win + 1, axis=0
    ).transpose(0, 2, 1)
    X_paired = np.lib.stride_tricks.sliding_window_view(
        windows, window_shape=win, axis=1
    ).transpose(0, 1, 3, 2)
    B_1 = B[win:]
    print(f'Prepped data, X shape {X_paired.shape}, B shape {B_1.shape}')
    return X_paired, B_1



def generate_B(start, neuron_traces_block):
    B = np.zeros(neuron_traces_block.shape[1], dtype=int)
    l = len(labels)

    for e in range(l):
        current_sample = max(0, labels[e][0] - start) 
        label = labels[e][1]

        if e + 1 < l:
            next_sample = min(len(B), labels[e + 1][0] - start)
        else:
            next_sample = len(B)

        if current_sample < next_sample:
            B[current_sample:next_sample] = label

    return B


q = queue.Queue(maxsize=2)

def fetch_data(): 
    for start in range(0, raw.n_times, block_size):
        stop = min(start+block_size, raw.n_times)
        neuron_traces_block = raw.get_data(picks = "eeg", start=start, stop=stop)
        print(f'Putting item into queue')
        q.put((start, neuron_traces_block))

def consume_data():
    while True:
        item = q.get()
        print(f'Working on {item}')
        print(f'Finished {item}')
        neuron_traces_block = item[1]
        B = generate_B(item[0], neuron_traces_block)
        print(f'Shape of generate B {B.shape}')
        prep_data(neuron_traces_block.T, B, win=win)
        q.task_done()

threading.Thread(target=fetch_data, daemon=True).start()
threading.Thread(target=consume_data, daemon=True).start()
q.join()
print("Finished)")

Lese BrainVision-Aufnahme ein: /home/bened/bachelor-thesis/data/raw/ds005363-download/sub-101/ses-01/eeg/sub-101_ses-01_task-objrecog_eeg.vhdr
Finished)


Putting item into queue
Working on (0, array([[-5.05e-05, -2.60e-05, -4.00e-06, ..., -1.90e-05, -2.15e-05,
        -2.50e-05],
       [-5.25e-05, -5.50e-05, -5.75e-05, ...,  2.90e-05,  2.90e-05,
         2.95e-05],
       [ 1.00e-05,  1.05e-05,  1.20e-05, ..., -2.35e-05, -2.35e-05,
        -2.40e-05],
       ...,
       [ 2.40e-05,  2.30e-05,  2.25e-05, ...,  4.50e-06,  4.50e-06,
         4.00e-06],
       [ 2.60e-05,  2.50e-05,  2.45e-05, ...,  5.50e-06,  5.00e-06,
         4.50e-06],
       [ 2.25e-05,  2.25e-05,  2.25e-05, ...,  7.50e-06,  7.00e-06,
         7.50e-06]], shape=(64, 16369)))
Finished (0, array([[-5.05e-05, -2.60e-05, -4.00e-06, ..., -1.90e-05, -2.15e-05,
        -2.50e-05],
       [-5.25e-05, -5.50e-05, -5.75e-05, ...,  2.90e-05,  2.90e-05,
         2.95e-05],
       [ 1.00e-05,  1.05e-05,  1.20e-05, ..., -2.35e-05, -2.35e-05,
        -2.40e-05],
       ...,
       [ 2.40e-05,  2.30e-05,  2.25e-05, ...,  4.50e-06,  4.50e-06,
         4.00e-06],
       [ 2.60e-05,  2.5

In [ ]:


# EEG-Kanäle auswählen
eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])

# Messwerte: (Kanäle, Zeitpunkte)
neuron_traces = raw.get_data(picks=eeg_picks)

# Kanalnamen in derselben Reihenfolge wie die Messwerte
channel_names = np.array(
    [raw.ch_names[i] for i in eeg_picks],
    dtype=object
)

# Abtastrate der Aufnahme: bei dir 1000 Hz
fps = raw.info["sfreq"]
state_names = [
    "keine Kategorie",
    "Gesicht",
    "Tier",
    "Ort",
    "Objekt",
    "Catch",
]

stimulus_to_state = {}

for code in range(1, 17):
    stimulus_to_state[code] = 1  # Gesicht

for code in range(17, 33):
    stimulus_to_state[code] = 2  # Tier

for code in range(33, 49):
    stimulus_to_state[code] = 3  # Ort

for code in range(49, 65):
    stimulus_to_state[code] = 4  # Objekt

for code in range(65, 70):
    stimulus_to_state[code] = 5
print("--------------------------------------- STIMULUS TO STATE ARRAY ---------------------------------------")
print(stimulus_to_state)
print("")
print("")

event = events[4] 

sample = int(event["sample"])
messwerte = neuron_traces[:, sample]

print("--------------------------------------- Neuron Traces ---------------------------------------")
print(neuron_traces[:,:])
print("")
print(neuron_traces[[0],:])
print("")
print(neuron_traces.shape)

print("--------------------------------------- EVENT NR. 4 ---------------------------------------")
print(event)
print("")
print("")
print("--------------------------------------- MESSWERTE FÜR SAMPLE VON EVENT NR. 4 ---------------------------------------")
print(messwerte)
print("")
print("")
print("--------------------------------------- FORM VON MESSWERTE ---------------------------------------")
print(messwerte.shape)
print("")
print("")
print("--------------------------------------- FORM VON NEURON TRACES ---------------------------------------")
print(neuron_traces.shape)
print("")
print("")
# print("--------------------------------------- FORM VON B ---------------------------------------")
# B = np.zeros(neuron_traces.shape[1], dtype=int)
# print(B.shape)

--------------------------------------- STIMULUS TO STATE ARRAY ---------------------------------------
{1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 1, 16: 1, 17: 2, 18: 2, 19: 2, 20: 2, 21: 2, 22: 2, 23: 2, 24: 2, 25: 2, 26: 2, 27: 2, 28: 2, 29: 2, 30: 2, 31: 2, 32: 2, 33: 3, 34: 3, 35: 3, 36: 3, 37: 3, 38: 3, 39: 3, 40: 3, 41: 3, 42: 3, 43: 3, 44: 3, 45: 3, 46: 3, 47: 3, 48: 3, 49: 4, 50: 4, 51: 4, 52: 4, 53: 4, 54: 4, 55: 4, 56: 4, 57: 4, 58: 4, 59: 4, 60: 4, 61: 4, 62: 4, 63: 4, 64: 4, 65: 5, 66: 5, 67: 5, 68: 5, 69: 5}


--------------------------------------- Neuron Traces ---------------------------------------
[[-5.050e-05 -2.600e-05 -4.000e-06 ...  6.900e-05  6.700e-05  6.550e-05]
 [-5.250e-05 -5.500e-05 -5.750e-05 ...  1.175e-04  1.165e-04  1.155e-04]
 [ 1.000e-05  1.050e-05  1.200e-05 ...  5.500e-06  2.500e-06  2.000e-06]
 ...
 [ 2.400e-05  2.300e-05  2.250e-05 ...  4.000e-06  3.500e-06  1.000e-06]
 [ 2.600e-05  2.500e-05  2.4